Hugging Face’s transformers library has two layers:
- **High‑level APIs**: things like pipeline`("text-generation")` that abstract away details.
- **Lower‑level APIs**: direct access to the model classes (`AutoModelForCausalLM`, `AutoTokenizer`, etc.) that wrap raw PyTorch code.

Using the lower‑level API means you’re closer to the actual neural network — you control tokenization, batching, logits, and decoding yourself.

**Models that wrap PyTorch** Hugging Face doesn’t reinvent the wheel; it **builds wrappers** around `PyTorch` implementations of `transformer architectures (like Llama, BERT, GPT)`. These wrappers make it easier to load pretrained `weights`, handle special `tokens`, and integrate with training/inference pipelines.

NVIDIA T4 GPUs mid‑range GPU (16 GB VRAM) that’s well‑suited for inference and light fine‑tuning of medium‑sized models (like Llama‑7B or 8B with quantization).

#### High‑level API (easy, abstracted)

```python
from transformers import pipeline

# High-level: just call pipeline
sentiment_analyzer = pipeline("sentiment-analysis")

result = sentiment_analyzer("I love working with transformers!")
print(result)
```

#### Lower‑level API (manual control)

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load tokenizer and model explicitly
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

# Step 1: Tokenize input
inputs = tokenizer("I love working with transformers!", return_tensors="pt")

# Step 2: Run through model
outputs = model(**inputs)

# Step 3: Convert logits to probabilities
probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

# Step 4: Interpret result
labels = model.config.id2label
predicted_label = labels[torch.argmax(probs).item()]
confidence = torch.max(probs).item()

print(f"Sentiment: {predicted_label}, Confidence: {confidence:.4f}")
```

![Comparison](image-6.png)

Machine Learning Engineer, the lower‑level API is where you’ll spend time when building custom RAG pipelines, LoRA fine‑tunes, or agent workflows.

#### Two versions of Llama instruct models hosted on Hugging Face

1. Llama 3.1 Instruct (8B)
    - A larger model (8 billion parameters). instructions and chat‑style prompts.
    - 8B is better for complex reasoning and richer outputs.
2. Llama 3.2 Instruct (1B)
    - A smaller model (1 billion parameters). Might need to request access again from Meta via Hugging Face.
    - 1B is better for lightweight inference, mobile/edge devices, or quick prototyping.
    
**How to know which model to use**

![How we know that which models are require](image-7.png)

👉 Since you’re aiming to build AI agents, RAG systems, and LoRA fine‑tunes, you’ll likely want 3.1 (8B) for richer reasoning. But if you’re experimenting with deployment on smaller hardware or want a lightweight demo, 3.2 (1B) is the practical choice.

####  gaining access to Meta’s Llama 3.1 models through Hugging Face:

TODO



## Instruct models and Reasoning models

#### Instruct models

Fine‑tuned versions of base LLMs that are trained to follow human instructions in a conversational or assistant‑like way. They expect prompts structured with roles (system, user, assistant) and respond in a helpful, aligned manner. There use cases Use cases: Chatbots and copilots, Summarization, Q&A, Code generation, RAG pipelines and AI agents.

```python
LLAMA = meta-llama/Meta-Llama-3.1-8B-Instruct
PHI = microsoft/Phi-4-mini-instruct
GEMMA = google/gemma-3-270m-it
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
```

#### Reasoning Models:
Models specifically optimized for step‑by‑step logical reasoning and problem solving. They generate intermediate reasoning steps (sometimes hidden, sometimes visible) before giving a final answer. Use cases: Math problem solving, Multi‑step logical tasks, Complex decision making, Scientific or technical reasoning.

```python
deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B (a distilled reasoning model)

```
![Instruct models and Reasoning models](image-8.png)

Use `Instruct models` when you want an assistant that follows instructions and interacts naturally. Use `Reasoning models` when you need deeper logical problem solving or multi‑step thinking. For building `AI agents and RAG systems`, you’ll often combine them:
- Instruct models for interaction and task execution.
- Reasoning models for complex decision‑making inside the agent pipeline.


## Accessing Llama 3.1 from Meta

TODO

## Quantization 

#### why it matters for running large models on GPUs with limited VRAM (like your 16 GB card)?

Quantization is the process of reducing the precision of the numbers (weights) stored in a model. Normally, model weights are stored in FP16 (16‑bit floating point) or FP32 (32‑bit floating point). Quantization compresses them into lower‑bit formats (like 8‑bit or 4‑bit integers).

- This reduces memory usage and speeds up inference, at the cost of a small accuracy trade‑off. quantization trades off a little accuracy for big memory savings.

In quantization model’s weights are normally stored in `FP16 (16‑bit) or FP32 (32‑bit)` floating point.
**Quantization** compresses them into **INT8 (8‑bit) or INT4 (4‑bit)** integers. ***This reduces VRAM usage by 50–75%, but the numbers are less precise***.

Example:
- Llama‑13B FP16 → needs ~26 GB VRAM.
- Llama‑13B INT4 → fits in ~8–10 GB VRAM.
- Accuracy difference: maybe a fraction of a percent in benchmarks (e.g., 68% → 66%).
- For most real‑world tasks (chat, summarization, RAG), the difference is barely noticeable.

Rule of Thumb:
- 7B models → run fine without quantization on 16 GB GPUs.
- 13B models → quantization recommended, accuracy loss is tiny.
- 32B models → quantization required, accuracy loss noticeable but still usable.
- 70B models → quantization + multi‑GPU/cloud only.

![Precision Levels](image-9.png)

#### Example (BitsAndBytesConfig):

```python
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                  # use 4-bit quantization
    bnb_4bit_use_double_quant=True,     # extra compression trick
    bnb_4bit_compute_dtype=torch.bfloat16, # compute in bfloat16 for stability
    bnb_4bit_quant_type="nf4"           # NormalFloat4, best for LLMs
)
```



## Padding tokens in LLMs

#### Table of Contents

1. [What is a Padding Token?](#1-what-is-a-padding-token)
2. [Why Padding Tokens are "Meaningless"](#2-why-padding-tokens-are-meaningless)
3. [What is an Attention Mask?](#3-what-is-an-attention-mask)
4. [How Attention Mask Works Internally](#4-how-attention-mask-works-internally)
5. [CNN Padding vs LLM Padding](#5-cnn-padding-vs-llm-padding)
6. [Code Examples](#6-code-examples)
7. [Summary Table](#7-summary-table)
8. [Quick Revision Q&A](#8-quick-revision-qa)

## 1. What is a Padding Token?

### The Problem

LLMs process inputs in **batches**. Each sequence in a batch can have a different length. But matrix operations require **uniform tensor shapes** — all sequences must be the same length.

### The Solution

A **padding token** (`[PAD]`) is a special dummy token appended (or prepended) to shorter sequences to make them match the length of the longest sequence in the batch.

```text
Sequence 1: ["The", "cat", "sat"]       -> length 3
Sequence 2: ["Hello", "world"]          -> length 2
```

After padding to length 3:
```text
Sequence 1: ["The", "cat", "sat" ]
Sequence 2: ["Hello", "world", "[PAD]"]
```

### Key Details

| Aspect | Detail |
| :--- | :--- |
| **Token ID** | Usually `'0'`, but model-specific (e.g., GPT-2 uses `'EOS'` as pad) |
| **Position** | Right-padding for encoder models (BERT), left-padding for decoder/causal models (GPT family) |
| **Why left for causal LLMs** | The model generates from the last real token — right-padding would shift the generation start position |
| **During training** | Loss is masked over padding positions so they don't contribute to gradient updates |
| **During inference** | Attention mask passed alongside `input_ids` to the model |

## 2. Why Padding Tokens are "Meaningless"

When you pad a sequence, you're adding **fake tokens that carry no real information**.  
They exist only to fill space. The model has no reason to learn anything from them or attend to them.

If the model **does** attend to them, it would be computing relationships between real words and fake filler — which **corrupts the output**.

```text
Real sentence: "The cat sat"
Padded:        "The cat sat [PAD] [PAD]"
```

What does "cat" have to do with `[PAD]`? — Nothing.  
What does `[PAD]` mean semantically? — Nothing.

> **Meaningless** does not mean the token doesn't exist.  
> It means **attending to it produces noise, not signal.**

## 3. What is an Attention Mask?

An attention mask is a **binary tensor** that tells the model:
- `1` — **pay attention** to this token (real token)
- `0` — **ignore** this token (padding token)

```text
Input:           ["The",  "cat",  "sat",  "[PAD]", "[PAD]"]
Token IDs:       [ 101,    5765,   2938,   0,       0      ]
Attention Mask:  [ 1,      1,      1,      0,       0      ]
```

### Visual Attention Map

```text
         The   cat   sat   [PAD] [PAD]
The     [ ✓     ✓     ✓      X     X  ]
cat     [ ✓     ✓     ✓      X     X  ]
sat     [ ✓     ✓     ✓      X     X  ]
[PAD]   [ X     X     X      X     X  ]  <- entire row blocked
[PAD]   [ X     X     X      X     X  ]  <- entire row blocked
```

Real tokens only attend to real tokens. Padding attends to nothing.

## 4. How Attention Mask Works Internally

In the **self-attention mechanism**, each token computes a score against every other token:

$$\text{Attention Score} = \text{softmax}\left(\frac{Q \times K^T}{\sqrt{d_k}}\right)$$

To mask out padding positions, their scores are set to **negative infinity** before softmax:

```text
Before masking: [0.8,  0.6,  0.4,   0.3,   0.2]
After masking:  [0.8,  0.6,  0.4,  -inf,  -inf]
After softmax:  [0.45, 0.35, 0.20,  0.0,   0.0]

                                     |      |
                                     effectively zero - ignored
```

So padding tokens get **zero attention weight** — the model never "sees" them.

### Step-by-Step Flow

```text
Input IDs  ──>  Embedding Layer
                     │
                     ▼
             Self-Attention (Q, K, V)
                     │
                     ▼
             Attention Scores (Q * K^T / √d)
                     │
                     ▼
             Apply Mask: set PAD positions -> -inf
                     │
                     ▼
                  Softmax
                     │
                     ▼
             PAD positions -> 0 weight
```

## 5. CNN Padding vs LLM Padding

> **Question asked:** Is this padding the same as what we use in CNN architecture with images?

### Side-by-Side Comparison

| Aspect | CNN Padding (Images) | LLM Padding (Text) |
| :--- | :--- | :--- |
| **Why** | Preserve spatial dimensions, control feature map size | Make sequences equal length for batch processing |
| **What is padded** | Pixel borders of an image (usually zeros) | End/start of a token sequence (dummy tokens) |
| **Content** | Zero pixel values | Special token ID (e.g., `[PAD]`) |
| **Used during computation?** | Yes — convolution slides over it | No — attention mask explicitly ignores it |
| **Affects output shape?** | Yes — 'same' padding keeps H×W unchanged | No — model ignores pad positions |

### CNN Padding - Spatial / Structural

Original $4\times4$ image with zero-padding ($\text{pad}=1$) $\rightarrow$ $6\times6$:

```text
0  0  0  0  0  0
0  1  2  3  4  0
0  5  6  7  8  0
0  9 10 11 12  0
0 13 14 15 16  0
0  0  0  0  0  0
```

- Padding **participates** in the convolution operation.
- It's about **geometry** — keeping edge pixels from losing context.

### LLM Padding - Batch Alignment / Administrative

```text
["Hello", "world", "[PAD]"]
attention_mask = [1, 1, 0]  -- model is told to SKIP [PAD]
```

- Padding is **explicitly ignored** by the model.
- It's about **infrastructure** — fitting variable-length sequences into a fixed tensor.

### The Analogy

| CNN Padding | LLM Padding |
| :--- | :--- |
| Margins on a page that the **printer still scans** | Blank pages at the end of a book that the **reader skips** |
| **Active filler** — convolution uses it | **Inactive placeholder** — attention ignores it |

> **Short answer:** Same high-level idea (fill to uniform shape), but CNN padding is  
> structurally meaningful while LLM padding is deliberately invisible to the model.

## 6. Code Examples

### Basic Tokenizer Padding (HuggingFace)

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

batch = ["The cat sat", "Hello world"]

encoded = tokenizer(
    batch,
    padding=True,          # pad to longest in batch
    return_tensors="pt"
)

print(encoded["input_ids"])
# tensor([[ 101, 1996, 4937, 2938,  102],
#         [ 101, 7592,  102,    0,    0]]) -> 0 = [PAD]

print(encoded["attention_mask"])
# tensor([[1, 1, 1, 1, 1],
#         [1, 1, 1, 0, 0]]) -> 0 = ignore
```

### Passing to Model

```python
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")

output = model(
    input_ids=encoded["input_ids"],
    attention_mask=encoded["attention_mask"]  # mask passed here
)
```

### Fix for GPT-2 (No Default Pad Token)

```python
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# GPT-2 has no dedicated pad token - set it manually
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# Now padding works correctly
encoded = tokenizer(["Hello world", "Hi"], padding=True, return_tensors="pt")
```

### Left Padding for Decoder-Only Models

```python
tokenizer.padding_side = "left"  # for GPT-style / causal LLMs

# Why left padding?
# Model generates from the last real token position.
# Right padding would push generation start to the wrong index.

# RIGHT padding (wrong for causal LM):
# ["Hello", "world", "[PAD]", "[PAD]"] -> model generates after [PAD]?

# LEFT padding (correct for causal LM):
# ["[PAD]", "[PAD]", "Hello", "world"] -> model generates after "world" ✓
```

### Manual Attention Mask (PyTorch)

```python
import torch

# Manually construct mask
input_ids = torch.tensor([,  # sequence 1 with 2 pads
    [101, 7592,  102,   0,   0,   0]   # sequence 2 with 3 pads
])

attention_mask = (input_ids != 0).long()  # 1 where not PAD, 0 where PAD

print(attention_mask)
# tensor([[1, 1, 1, 1, 0, 0],
#         [1, 1, 1, 0, 0, 0]])
```

## 7. Summary Table

| Concept | Role |
| :--- | :--- |
| **Padding token `[PAD]`** | Dummy token filling empty space in a batch |
| **Token ID** | Usually `0`; model specific |
| **Meaningless** | Has no semantic content — attending to it adds noise |
| **Attention mask** | Binary filter: `1` = attend, `0` = ignore |
| **Mechanism** | Sets padding positions to `-inf` before softmax $\rightarrow$ zero weight |
| **CNN padding** | Active filler — convolution uses it for spatial geometry |
| **LLM padding** | Inactive placeholder — model explicitly ignores it |
| **Left vs Right padding** | Left for decoder/causal LLMs; Right for encoder models |

## 8. Quick Revision Q&A

**Q1: Why do we need padding tokens?**
> To batch sequences of different lengths into a single tensor for efficient GPU processing.

**Q2: Why are padding tokens called "meaningless"?**
> They carry no semantic information. Attending to them introduces noise into the model's computation.

**Q3: What does an attention mask do?**
> It tells the model which positions to attend to (`1`) and which to ignore (`0`), by setting ignored positions to `-inf` before the softmax operation.

**Q4: Is CNN padding the same as LLM padding?**
> Similar concept (filling space for uniform shape) but different purpose. CNN padding is active — convolution uses it. LLM padding is inactive — the model is explicitly told to skip it.

**Q5: Why does GPT-2 need a special fix for padding?**
> GPT-2 was trained without a dedicated `[PAD]` token. You must manually assign one (usually `EOS`) before using it in batched inference.

**Q6: Why do causal LLMs use left padding instead of right?**
> Decoder models generate from the last token position. Left padding ensures the last real token is always at the rightmost position, so generation starts correctly.

### Example With Quantization

#### 1. Tokenizer Setup
```python
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
# Sets the padding token to the same as the end‑of‑sequence token (so padding doesn’t confuse the model).
tokenizer.pad_token = tokenizer.eos_token
# Converts your structured messages (system/user roles) into a proper input tensor using apply_chat_template.
# This prepares the input prompt in the correct format for the model.
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
```

#### 2. Model Loading with Quantization
```python
model = AutoModelForCausalLM.from_pretrained(
    LLAMA, # Loads the causal language model (LLM) for text generation.
    device_map="auto",# automatically spreads the model across available devices (GPU/CPU)

    # applies quantization (like 4‑bit NF4) to reduce memory usage.
    # This ensures the model fits into limited VRAM (e.g., your 16 GB GPU).
    quantization_config=quant_config 
)
```

#### 3. Memory Footprint Calculation:

```python
# Calls get_memory_footprint() to measure how much GPU memory the model is using.
# Divides by 1e6 to convert bytes → MB.
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:.1f} MB")
```

This workflow is super useful when you’re testing whether a model fits into your GPU VRAM. For example: 
- A 13B model in FP16 might exceed 16 GB.
- With 4‑bit quantization, the footprint drops to ~8–10 GB, which fits.

## VRAM monitoring:



#### Load Model Without Quantization

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-hf")
inputs = tokenizer("Hello world!", return_tensors="pt").to("cuda")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-13b-hf",
    device_map="auto"
)

print(f"FP16 footprint: {model_fp16.get_memory_footprint()/1e9:.2f} GB")


#### Apply Quantization (INT8 or INT4)

In [ ]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype="bfloat16",
    bnb_4bit_quant_type="nf4"
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-13b-hf",
    device_map="auto",
    quantization_config=quant_config
)

print(f"INT4 footprint: {model_int4.get_memory_footprint()/1e9:.2f} GB")

#### Decision Logic

In [ ]:
gpu_vram = 16  # your GPU VRAM in GB

if model_int4.get_memory_footprint()/1e9 < gpu_vram:
    print("✅ Model fits in GPU with quantization")
else:
    print("❌ Model too large, consider offloading or smaller model")


#### Understanding the Llama 3.1 Transformer Architecture

[Understanding the Llama 3.1 Transformer Architecture](https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b)

## PyTorch neural network

TODO

port youe CNN models on pytorch

[HuggingFace Transformers repo](https://github.com/huggingface/transformers)

[code for Llama 4](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py)

“When you load Llama in Hugging Face, you’re actually loading a Transformer neural network with embeddings, decoder layers, and an LM head. It’s quantized to 4‑bit so it fits in memory. Take a look at the printed layers to understand the architecture under the hood.”

Model object = Neural Network

When you load `AutoModelForCausalLM` in Hugging Face, what you actually get is a PyTorch neural network. This network is built on the Transformer architecture (introduced by Google in 2017).

#### what is TextStreamer  actually do?

`TextStreamer` is a utility in Hugging Face Transformers that controls how generated tokens are returned during inference. Instead of waiting for the model to finish generating the entire output, it streams tokens one by one (or in small chunks) back to you in real time.

##### What TextStreamer Actually Does ?

- **Intercepts tokens**: As the model generates each new token, TextStreamer captures it immediately.
- **Prints/returns progressively**: It sends the token to the console or output stream right away, so you see text appearing like a chatbot typing.
- **Manages formatting**: It uses the tokenizer to decode tokens into readable text before displaying them.
- **Improves UX**: You don’t wait for the full response — you see partial results as they’re produced.

TextStreamer is like a live feed of the model’s output — instead of waiting for the full answer, you get tokens streamed in real time, decoded into text as they arrive.

## Accessing Gemma from Google